
# NDX Garden — Workbench
### Module 3: Regression and Core Optimization — Hands-On

Welcome. In the lesson you saw the math and the interactive components — here you'll write the actual code: fit real models, compute real diagnostics, and implement one algorithm (Gradient Descent) entirely by hand.

Each exercise has a task cell for you to fill in, marked `# YOUR CODE HERE`. A full **Solutions** section sits at the bottom of this notebook so you can self-check — try each exercise yourself before peeking.

**Note:** this notebook is a supplementary practice space, separate from your progress in the main NDX Garden lesson. Nothing here is graded or tracked.



## Setup: the dataset

We're using a synthetic housing dataset, generated below, with a deliberate structure:
- `sqft` — genuinely predictive of price
- `sqft_correlated` — a second feature almost identical to `sqft` (this is our multicollinearity case — same naming as the `RegularizationGeometryDiagram` component in the lesson)
- `bedrooms` — mildly predictive
- `noise_feature` — pure random noise, not predictive at all

Run the cell below to generate and inspect it.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools import add_constant
from sklearn.preprocessing import PolynomialFeatures

np.random.seed(13)
n = 200

sqft = np.random.normal(1800, 500, n)
sqft_correlated = sqft + np.random.normal(0, 50, n)   # near-duplicate of sqft
bedrooms = np.round(sqft / 700 + np.random.normal(0, 0.5, n)).clip(1, 6)
noise_feature = np.random.normal(0, 1, n)

price = (
    150 * sqft
    + 5000 * bedrooms
    + 20000
    + np.random.normal(0, 25000, n)   # irreducible noise
)

df = pd.DataFrame({
    "sqft": sqft,
    "sqft_correlated": sqft_correlated,
    "bedrooms": bedrooms,
    "noise_feature": noise_feature,
    "price": price,
})

df.head()



## Exercise 1 — Fit `LinearRegression` and check the assumptions

1. Split `df` into `X` (all columns except `price`) and `y` (`price`).
2. Split into train/test (80/20, `random_state=42`).
3. Fit `LinearRegression` on the training data.
4. Print the intercept and coefficients.
5. Plot residuals (predicted vs. residual) on the **test** set — this is the same diagnostic plot from `AssumptionDiagnosticsDiagram` in the lesson. Does it look "Healthy," or do you see a pattern?


In [ ]:

# YOUR CODE HERE
X = None
y = None

X_train, X_test, y_train, y_test = None, None, None, None

model = None

# print intercept and coefficients

# plot residuals: predicted (x-axis) vs residual = actual - predicted (y-axis)



## Exercise 2 — Compute VIF for each feature

Use `variance_inflation_factor` from `statsmodels` to compute the VIF for every feature in `X`. Remember: VIF needs a constant column added first (statsmodels' `add_constant`), since it's regressing each feature against all the others.

Which feature(s) have a high VIF? Does it match what you'd expect given how the dataset was built?


In [ ]:

# YOUR CODE HERE
X_with_const = None  # add_constant(X)

vif_data = None  # build a small DataFrame: feature name -> VIF value



## Exercise 3 — Implement Gradient Descent by hand

This is the one exercise where writing the loop yourself matters more than calling a library. Using **only `sqft`** as a single feature (to keep it simple), implement Batch Gradient Descent from scratch:

1. Standardize `sqft` and `price` (subtract mean, divide by std) — this keeps the learning rate well-behaved.
2. Initialize `theta0 = 0`, `theta1 = 0`.
3. For a fixed number of iterations, compute the gradient of the MSE cost with respect to `theta0` and `theta1` (same formula as the lesson's `gradient-descent` screen), and update both parameters.
4. Track the cost at every iteration and plot it — it should decrease and flatten out.
5. Compare your final `theta0`/`theta1` (on standardized data) against what `LinearRegression` finds on the same standardized single-feature data.


In [ ]:

# YOUR CODE HERE
x = None  # standardized sqft
y_std = None  # standardized price

theta0, theta1 = 0.0, 0.0
alpha = 0.1
n_iterations = 200
costs = []

for i in range(n_iterations):
    pass  # compute predictions, gradients, update theta0/theta1, record cost

# plot costs vs iteration

# compare to LinearRegression on the same standardized single feature



## Exercise 4 — Ridge, Lasso, Elastic Net across a range of alpha

1. For `alpha` values `[1, 10, 100, 1000, 3000, 10000]`, fit a `Ridge` model (inside a `StandardScaler` pipeline, same pattern as the lesson) on the training data.
2. Record each feature's coefficient at each `alpha`.
3. Plot each feature's coefficient vs. `alpha` (log x-axis) — this is a live version of the lesson's `RegularizationGeometryDiagram` Panel B, but showing the full shrinkage path instead of one snapshot.
4. Repeat for `Lasso`. At what `alpha` does `noise_feature` (the genuinely irrelevant feature) hit exactly zero? Does `sqft_correlated` (merely *redundant* with `sqft`, not irrelevant) zero out anywhere in this range too, or does it take much stronger regularization to knock out? What does that difference tell you about how Lasso treats "irrelevant" vs. "redundant" features?


In [ ]:

# YOUR CODE HERE
alphas = [1, 10, 100, 1000, 3000, 10000]

ridge_coefs = []  # list of coefficient arrays, one per alpha
lasso_coefs = []

for a in alphas:
    pass  # fit Ridge(alpha=a) and Lasso(alpha=a) inside a StandardScaler pipeline, store .coef_

# plot ridge_coefs vs alphas (log x-axis), one line per feature
# plot lasso_coefs vs alphas (log x-axis), one line per feature



## Exercise 5 (stretch) — Find your own polynomial sweet spot

This one uses a **different, dedicated toy dataset** (generated below) rather than the housing data above — the housing price relationship is close to linear by construction, so it doesn't actually show a real overfitting signal at high polynomial degree. The dataset below has genuine curvature and few enough points that high-degree fits will visibly chase noise, the same design tension the lesson's `PolynomialDegreeExplorer` deals with.

1. For degrees 1 through 12, expand `x` into polynomial features (`PolynomialFeatures`), fit a `LinearRegression` on the training split, and record train and test MSE.
2. Plot train error and test error vs. degree — this is a coded version of the lesson's `PolynomialDegreeExplorer`.
3. Where does test error bottom out? What happens to the gap between train and test error as degree keeps increasing past that point?


In [ ]:

# Dedicated toy dataset for this exercise (run this cell, then write your code below)
np.random.seed(7)
n_poly = 18
x_poly = np.linspace(-3.5, 3.5, n_poly)
true_y = 0.15 * x_poly**3 - 0.8 * x_poly**2 + 0.5 * x_poly + 2
y_poly = true_y + np.random.normal(0, 1.2, n_poly)

Xp = x_poly.reshape(-1, 1)
Xp_train, Xp_test, yp_train, yp_test = train_test_split(Xp, y_poly, test_size=5, random_state=7)


In [ ]:

# YOUR CODE HERE
degrees = range(1, 13)
train_errors = []
test_errors = []

for d in degrees:
    pass  # expand Xp_train/Xp_test to degree d, fit, record train/test MSE

# plot train_errors and test_errors vs degrees



---
## Solutions

Try every exercise yourself first. These are full worked solutions for self-checking.


### Solution 1

In [ ]:

X = df.drop(columns="price")
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

print("Intercept:", model.intercept_)
print("Coefficients:")
for name, coef in zip(X.columns, model.coef_):
    print(f"  {name}: {coef:.3f}")

y_pred_test = model.predict(X_test)
residuals = y_test - y_pred_test

plt.figure(figsize=(6, 4))
plt.scatter(y_pred_test, residuals, alpha=0.6)
plt.axhline(0, color="gray", linestyle="--")
plt.xlabel("Predicted price")
plt.ylabel("Residual")
plt.title("Residuals vs. Predicted (test set)")
plt.show()


### Solution 2

In [ ]:

X_with_const = add_constant(X)

vif_data = pd.DataFrame({
    "feature": X_with_const.columns,
    "VIF": [variance_inflation_factor(X_with_const.values, i) for i in range(X_with_const.shape[1])]
})
vif_data = vif_data[vif_data["feature"] != "const"]
print(vif_data)


### Solution 3

In [ ]:

x = (df["sqft"] - df["sqft"].mean()) / df["sqft"].std()
y_std = (df["price"] - df["price"].mean()) / df["price"].std()

theta0, theta1 = 0.0, 0.0
alpha = 0.1
n_iterations = 200
costs = []
n_pts = len(x)

for i in range(n_iterations):
    y_pred = theta0 + theta1 * x
    error = y_pred - y_std
    cost = (error ** 2).mean()
    costs.append(cost)
    grad0 = (2 / n_pts) * error.sum()
    grad1 = (2 / n_pts) * (error * x).sum()
    theta0 -= alpha * grad0
    theta1 -= alpha * grad1

plt.figure(figsize=(6, 4))
plt.plot(costs)
plt.xlabel("Iteration")
plt.ylabel("Cost (MSE)")
plt.title("Gradient Descent convergence")
plt.show()

print(f"Hand-rolled GD:   theta0={theta0:.4f}, theta1={theta1:.4f}")

sklearn_check = LinearRegression().fit(x.values.reshape(-1, 1), y_std)
print(f"LinearRegression: theta0={sklearn_check.intercept_:.4f}, theta1={sklearn_check.coef_[0]:.4f}")


### Solution 4

In [ ]:

alphas = [1, 10, 100, 1000, 3000, 10000]
feature_names = X_train.columns

ridge_coefs = []
lasso_coefs = []

for a in alphas:
    ridge_pipe = make_pipeline(StandardScaler(), Ridge(alpha=a)).fit(X_train, y_train)
    lasso_pipe = make_pipeline(StandardScaler(), Lasso(alpha=a, max_iter=10000)).fit(X_train, y_train)
    ridge_coefs.append(ridge_pipe.named_steps["ridge"].coef_)
    lasso_coefs.append(lasso_pipe.named_steps["lasso"].coef_)

ridge_coefs = np.array(ridge_coefs)
lasso_coefs = np.array(lasso_coefs)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for j, name in enumerate(feature_names):
    axes[0].plot(alphas, ridge_coefs[:, j], marker="o", label=name)
    axes[1].plot(alphas, lasso_coefs[:, j], marker="o", label=name)

for ax, title in zip(axes, ["Ridge", "Lasso"]):
    ax.set_xscale("log")
    ax.set_xlabel("alpha (log scale)")
    ax.set_ylabel("coefficient")
    ax.set_title(title)
    ax.legend()
    ax.axhline(0, color="gray", linewidth=0.5)

plt.tight_layout()
plt.show()

print("Lasso coefficients across alpha (rows=alpha, cols=features):")
print(pd.DataFrame(lasso_coefs, index=alphas, columns=feature_names))


### Solution 5

In [ ]:

degrees = range(1, 13)
train_errors = []
test_errors = []

for d in degrees:
    poly = PolynomialFeatures(degree=d)
    Xpd_train = poly.fit_transform(Xp_train)
    Xpd_test = poly.transform(Xp_test)

    m = LinearRegression().fit(Xpd_train, yp_train)
    train_errors.append(mean_squared_error(yp_train, m.predict(Xpd_train)))
    test_errors.append(mean_squared_error(yp_test, m.predict(Xpd_test)))

plt.figure(figsize=(6, 4))
plt.plot(list(degrees), train_errors, marker="o", label="Train MSE")
plt.plot(list(degrees), test_errors, marker="o", label="Test MSE")
plt.yscale("log")
plt.xlabel("Polynomial degree")
plt.ylabel("MSE (log scale)")
plt.title("Train vs. Test error by degree")
plt.legend()
plt.show()

best_degree = list(degrees)[int(np.argmin(test_errors))]
print(f"Lowest test error at degree: {best_degree}")
print("Train error keeps falling (and eventually hits ~0) as degree rises — but test error bottoms out "
      "around the true cubic's degree, then climbs sharply. That widening train/test gap at high degree "
      "is overfitting, visible directly in the numbers.")



---
*NDX Garden — Module 3 Workbench. Back to [projectndx.com](https://projectndx.com) · Next: Module 4 Workbench (coming soon)*
